# Traffic Demand Forecasting - LightGBM + CatBoost Version

In [43]:
!pip install -q catboost lightgbm pygeohash

In [44]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pygeohash as pgh

from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [45]:
TRAIN_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/train.csv"
TEST_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [46]:
def create_features(df):

    df = df.copy()

    # ------------------------
    # TIME FEATURES
    # ------------------------

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="%H:%M",
        errors="coerce"
    )

    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute

    df["time_minutes"] = (
        df["hour"] * 60 +
        df["minute"]
    )

    df["hour_sin"] = np.sin(
        2*np.pi*df["hour"]/24
    )

    df["hour_cos"] = np.cos(
        2*np.pi*df["hour"]/24
    )

    # ------------------------
    # GEOHASH FEATURES
    # ------------------------

    latitudes = []
    longitudes = []

    for geo in df["geohash"]:

        try:
            lat, lon = pgh.decode(geo)

            latitudes.append(float(lat))
            longitudes.append(float(lon))

        except:
            latitudes.append(np.nan)
            longitudes.append(np.nan)

    df["latitude"] = latitudes
    df["longitude"] = longitudes

    # ------------------------
    # FEATURE INTERACTIONS
    # ------------------------

    df["lane_temp"] = (
        df["NumberofLanes"] *
        df["Temperature"]
    )

    df["temp_sq"] = (
        df["Temperature"] ** 2
    )

    return df

train = create_features(train)
test = create_features(test)

In [47]:
center_lat = train["latitude"].mean()
center_lon = train["longitude"].mean()

for df in [train, test]:

    df["dist_center"] = np.sqrt(
        (df["latitude"] - center_lat)**2 +
        (df["longitude"] - center_lon)**2
    )

In [48]:
freq = train["geohash"].value_counts()

train["geohash_freq"] = train["geohash"].map(freq)

test["geohash_freq"] = (
    test["geohash"]
    .map(freq)
    .fillna(0)
)

In [49]:
coords = pd.concat([
    train[["latitude","longitude"]],
    test[["latitude","longitude"]]
])

coords = coords.fillna(coords.median())

kmeans = KMeans(
    n_clusters=50,
    random_state=42,
    n_init=20
)

kmeans.fit(coords)

train["location_cluster"] = kmeans.predict(
    train[["latitude","longitude"]]
    .fillna(coords.median())
)

test["location_cluster"] = kmeans.predict(
    test[["latitude","longitude"]]
    .fillna(coords.median())
)

In [50]:
cat_cols = [
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "geohash"
]

for col in cat_cols:

    train[col] = (
        train[col]
        .fillna("Unknown")
        .astype(str)
    )

    test[col] = (
        test[col]
        .fillna("Unknown")
        .astype(str)
    )

In [51]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le.fit(combined)

    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

In [52]:
TARGET = "demand"

test_ids = test["Index"]

X = train.drop(columns=[TARGET])
y = train[TARGET]

X = X.drop(columns=["Index", "timestamp"])
X_test = test.drop(columns=["Index", "timestamp"])

In [53]:
for col in X.columns:

    median_val = X[col].median()

    X[col] = X[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

In [54]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_preds = np.zeros(len(X_test))
lgb_preds = np.zeros(len(X_test))

In [55]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):

    print(f"\nFold {fold+1}")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    # ------------------------
    # CATBOOST
    # ------------------------

    cat_model = CatBoostRegressor(
        iterations=3500,
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=8,
        random_strength=2,
        loss_function="RMSE",
        task_type="GPU",
        devices="0",
        random_seed=42,
        verbose=False
    )

    cat_model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=300
    )

    cat_preds += (
        cat_model.predict(X_test)
        / kf.n_splits
    )

    # ------------------------
    # LIGHTGBM
    # ------------------------

    lgb_model = LGBMRegressor(
        n_estimators=3200,
        learning_rate=0.02,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    lgb_model.fit(
        X_train,
        y_train
    )

    lgb_preds += (
        lgb_model.predict(X_test)
        / kf.n_splits
    )


Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004206 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1701
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 20
[LightGBM] [Info] Start training from score 0.093784

Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1701
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 20
[LightGBM] [Info] Start training from score 0.093797

Fold 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011591 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1701
[LightGBM] [Info] Number of data

In [56]:
final_preds = (
    0.70 * cat_preds +
    0.30 * lgb_preds
)

In [57]:
submission = pd.DataFrame({
    "Index": test_ids,
    "demand": final_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,Index,demand
0,0,0.064078
1,1,0.028764
2,2,0.022716
3,3,0.033867
4,4,0.058021


In [58]:
from IPython.display import FileLink
FileLink("submission.csv")

/kaggle/working/submission.csv